# Colab 03 — Frozen ESM-2 cache and one advanced experiment

## Goal

Copy one processed dataset into `/content`, optionally build a reusable frozen residue-level ESM-2 cache, run one advanced model experiment, and sync durable artefacts to Drive.

**Embedding generation, training, and sync default to off.**


## Setup


In [ ]:
from pathlib import Path
from importlib import import_module
import json
import os
import shutil
import subprocess
import sys

GIT_URL = "https://github.com/Thebluesky303/msc-dissertation-drug-repurposing.git"
BRANCH = "v2-refactor"
REPO_DIR = Path("/content/msc-dissertation-drug-repurposing")
WORKSPACE_DIR = Path("/content/dti_workspace")
DRIVE_ROOT = Path("/content/drive/MyDrive/msc-dissertation-drug-repurposing")


def configure_workspace_environment() -> None:
    # Configure all paths before importing config or any src module.
    os.environ["AML_GNN_PROJECT_ROOT"] = str(REPO_DIR)
    os.environ["AML_GNN_DATA_DIR"] = str(WORKSPACE_DIR / "data")
    os.environ["AML_GNN_MODELS_DIR"] = str(WORKSPACE_DIR / "checkpoints")
    os.environ["AML_GNN_RESULTS_DIR"] = str(WORKSPACE_DIR / "results")
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))


def require_api(module_name: str, attribute: str):
    module = import_module(module_name)
    if not hasattr(module, attribute):
        raise RuntimeError(f"Missing required v2 API: {module_name}.{attribute}")
    return getattr(module, attribute)


def copy_directory(source: Path, destination: Path) -> None:
    if not source.exists():
        raise FileNotFoundError(source)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, destination, dirs_exist_ok=True)


print(json.dumps({
    "git_url": GIT_URL,
    "branch": BRANCH,
    "repo": str(REPO_DIR),
    "workspace": str(WORKSPACE_DIR),
    "drive_root": str(DRIVE_ROOT),
}, indent=2))


In [ ]:
DATASET = "davis"
SPLIT = "random"
SEED = 42
ESM2_MODEL_NAME = "facebook/esm2_t6_8M_UR50D"
ESM2_MODEL_REVISION = None  # set an immutable Hugging Face commit for final runs
CACHE_PATH = WORKSPACE_DIR / "caches" / "esm2" / f"{DATASET}_esm2_t6_8m_residues.pt"
CACHE_METADATA_PATH = CACHE_PATH.with_suffix(".metadata.json")

RUN_STAGE_DATA = False
RUN_EMBEDDING = False
RUN_TRAINING = False
RUN_SYNC_TO_DRIVE = False


## Steps


In [ ]:
if RUN_STAGE_DATA:
    copy_directory(DRIVE_ROOT / "data" / "raw" / DATASET, WORKSPACE_DIR / "data" / "raw" / DATASET)
    if (REPO_DIR / "data" / "splits").exists():
        copy_directory(REPO_DIR / "data" / "splits", WORKSPACE_DIR / "data" / "splits")
    if (DRIVE_ROOT / "data" / "splits").exists():
        copy_directory(DRIVE_ROOT / "data" / "splits", WORKSPACE_DIR / "data" / "splits")
    drive_cache = DRIVE_ROOT / "caches" / "esm2"
    if drive_cache.exists():
        copy_directory(drive_cache, WORKSPACE_DIR / "caches" / "esm2")
    if CACHE_PATH.exists() and not CACHE_METADATA_PATH.exists():
        raise FileNotFoundError(
            f"Staged cache is missing required sidecar: {CACHE_METADATA_PATH}"
        )
else:
    print("Data/cache staging skipped: RUN_STAGE_DATA is False.")


In [ ]:
if RUN_EMBEDDING:
    configure_workspace_environment()
    import pandas as pd

    prepare_dataset = require_api("src.preprocessing", "prepare_dataset")
    build_cache = require_api("src.preprocessing", "build_esm2_residue_embedding_cache")

    pairs, _ = prepare_dataset(
        dataset_name=DATASET,
        raw_data_directory=WORKSPACE_DIR / "data" / "raw" / DATASET,
        output_path=WORKSPACE_DIR / "data" / "processed" / f"{DATASET}_pairs.csv",
    )
    residue_embedding_cache = build_cache(
        dataframe=pairs,
        output_path=CACHE_PATH,
        model_name=ESM2_MODEL_NAME,
        model_revision=ESM2_MODEL_REVISION,
    )
    if not CACHE_METADATA_PATH.exists():
        raise FileNotFoundError(
            f"Cache builder did not create required sidecar: {CACHE_METADATA_PATH}"
        )
    print({
        "cached_targets": len(residue_embedding_cache),
        "cache_path": str(CACHE_PATH),
        "metadata_path": str(CACHE_METADATA_PATH),
    })
else:
    print("ESM-2 cache generation skipped: RUN_EMBEDDING is False.")


In [ ]:
experiment_record = None
if RUN_TRAINING:
    configure_workspace_environment()
    if not CACHE_METADATA_PATH.exists():
        raise FileNotFoundError(
            f"Required cache metadata is missing: {CACHE_METADATA_PATH}"
        )

    from config import ExperimentConfig, SPLITS_DIR
    load_cache = require_api("src.preprocessing", "load_residue_embedding_cache")
    run_advanced_experiment = require_api("src.training", "run_advanced_experiment")
    experiment = ExperimentConfig(
        dataset=DATASET, split=SPLIT, model="advanced", seed=SEED, batch_size=16
    )
    manifest_path = SPLITS_DIR / "manifests" / (
        f"{experiment.dataset}_{experiment.split}_seed{experiment.seed}_manifest.csv"
    )
    residue_embedding_cache = load_cache(CACHE_PATH)
    experiment_record = run_advanced_experiment(
        config=experiment,
        residue_embedding_cache=residue_embedding_cache,
        split_manifest_path=manifest_path,
    )
    print(json.dumps(experiment_record, indent=2, default=str))
else:
    print("Advanced training skipped: RUN_TRAINING is False.")


In [ ]:
if RUN_SYNC_TO_DRIVE:
    if experiment_record is None and not CACHE_PATH.exists():
        raise RuntimeError("No inspected cache or experiment output is available to sync.")
    if CACHE_PATH.exists():
        if not CACHE_METADATA_PATH.exists():
            raise FileNotFoundError(
                f"Refusing to sync cache without sidecar: {CACHE_METADATA_PATH}"
            )
        destination_directory = DRIVE_ROOT / "caches" / "esm2"
        destination_directory.mkdir(parents=True, exist_ok=True)
        for cache_artifact in (CACHE_PATH, CACHE_METADATA_PATH):
            shutil.copy2(
                cache_artifact,
                destination_directory / cache_artifact.name,
            )
    for name in ("checkpoints", "results"):
        source = WORKSPACE_DIR / name
        if source.exists():
            copy_directory(source, DRIVE_ROOT / name)
    if (WORKSPACE_DIR / "data" / "splits").exists():
        copy_directory(
            WORKSPACE_DIR / "data" / "splits",
            DRIVE_ROOT / "data" / "splits",
        )
    print("Durable advanced artefacts synced.")
else:
    print("Drive sync skipped: RUN_SYNC_TO_DRIVE is False.")


## Checks

Confirm GPU availability before expensive work, frozen ESM-2 parameters, cache metadata/model identity, valid masks, finite outputs, and matched split/seed/protocol before comparison.

## Next steps

Preserve the cache, its `.metadata.json` sidecar, and the checkpoint in Drive, then use the local comparison notebooks for interpretation.
